# Project Analisis Sentimen Ulasan Aplikasi Gojek
### 1. Import Library & Modul Utama
Mengimpor library yang dibutuhkan untuk manipulasi data, preprocessing teks, pembentukan model deep learning (Dense, BiLSTM, Conv1D), serta evaluasi metrik.

In [58]:
# mengimport library untuk membersihkan stopwords
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
print("Library berhasil di-import!")

Library berhasil di-import!


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [59]:
# mengimport tensorflow dan keras untuk pembuatan model NLP
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Embedding, Dense, Dropout, LSTM, Bidirectional,
                                     SpatialDropout1D, Conv1D, GlobalMaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
print("Library berhasil di-import!")

Library berhasil di-import!


In [60]:
# mengimport scikit learn
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
print("Library berhasil di-import!")

Library berhasil di-import!


### 2. Memuat Dataset dan Pelabelan Sentimen
Load dataset mentah hasil scraping dan dikategorikan ke dalam 3 kelas sentimen (*Negatif*, *Netral*, *Positif*) berdasarkan rating (score 1-2 = Negatif, 3 = Netral, 4-5 = Positif).

In [61]:
# load dataset hasil scraping data google play
df = pd.read_csv("dataset_gojek_raw.csv")
print(f"Total data awal: {len(df)}")

print("\nBerikut adalah gambaran datasetnya")
df.head()

Total data awal: 35000

Berikut adalah gambaran datasetnya


,content,score
0,aplikasi GO block driver terdekat juga banyak ...,1
1,pengiriman murah cuma ujung²nya ga dianter sur...,1
2,apaan mau up&rade pake foto ktp dan muka,1
3,"aplikasi aneh, ada gratis ongkir tapi suruh ba...",1
4,gojek sekarang kenapa ya untuk mencari cepat t...,1


In [62]:
# pelabelan data dari score
def label_sentimen(score):
    if score in [1, 2]:
        return 'Negatif'
    elif score == 3:
        return 'Netral'
    else:
        return 'Positif'

df['label'] = df['score'].apply(label_sentimen)
print("Distribusi Kelas Sentimen: ")
print(df['label'].value_counts())

Distribusi Kelas Sentimen: 
label
Negatif    14000
Positif    14000
Netral      7000
Name: count, dtype: int64


### 3. Preprocessing dan Pembersihan Teks
Tahap pembersihan meliputi penanganan slang/typo, pembersihan karakter khusus, pengisian spasi emoji, mempertimbangkan kata negasi & konteks netral, serta penghapusan data kontradiktif & duplikat.

In [63]:
# membersihkan teks menggunakan NLTK stopwords indonesia
kata_dipertahankan = {'tidak', 'bukan', 'kurang', 'jangan', 'belum',
                      'tapi', 'namun', 'cukup', 'lumayan', 'biasa'}

stop_words_custom = set(stopwords.words('indonesian')) - kata_dipertahankan

slang_mapping = {
    'ga': 'tidak', 'gak': 'tidak', 'gk': 'tidak', 'g': 'tidak', 'tdk': 'tidak',
    'nggak': 'tidak', 'enggak': 'tidak', 'kagak': 'tidak', 'ndak': 'tidak',
    'yg': 'yang', 'y': 'yang',
    'udah': 'sudah', 'udh': 'sudah', 'sdh': 'sudah', 'dpt': 'dapat', 'dapet': 'dapat',
    'aja': 'saja', 'aj': 'saja', 'jga': 'juga', 'jg': 'juga',
    'nunggu': 'tunggu', 'pake': 'pakai', 'pakek': 'pakai', 'paki': 'pakai',
    'pesen': 'pesan', 'psn': 'pesan',
    'apk': 'aplikasi', 'aplikasinya': 'aplikasi', 'app': 'aplikasi',
    'bgt': 'banget', 'bngt': 'banget', 'bgus': 'bagus', 'bgs': 'bagus',
    'kalo': 'kalau', 'klo': 'kalau', 'kl': 'kalau',
    'cancle': 'cancel', 'cancil': 'cancel', 'kebatal': 'batal',
    'dri': 'driver', 'drvr': 'driver', 'drivernya': 'driver', 'mitra': 'driver',
    'lg': 'lagi', 'lgi': 'lagi', 'mulu': 'selalu', 'trus': 'terus',
    'sy': 'saya', 'ak': 'aku', 'sya': 'saya',
    'ongkir': 'ongkos kirim', 'gofood': 'gofood', 'gopay': 'gopay', 'goride': 'goride', 'gocar': 'gocar',
    'mbalikin': 'kembalikan', 'balikin': 'kembalikan', 'sampe': 'sampai', 'smpe': 'sampai',
    'tp': 'tapi', 'tpi': 'tapi', 'krn': 'karena', 'karna': 'karena', 'dr': 'dari',
    'bikin': 'buat', 'sm': 'sama', 'stlh': 'setelah', 'lemot': 'lambat', 'cpet': 'cepat', 'cpt': 'cepat'
}

def clean_text(text):
    text = str(text).lower() # mengubah semua huruf ke huruf kecil
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text) # menghapus semua link url seperti HTTP dan HTTPS
    text = re.sub(r'(.)\1{2,}', r'\1\1', text) # menghapus huruf berulang
    text = re.sub(r'[^a-z0-9\s]', ' ', text) # menghapus semua simbol atau karakter selain huruf dan angka

    # memecah teks menjadi daftar kata
    words = text.split()
    normalized = [slang_mapping.get(w, w) for w in words] # mengubah kata gaul menjadi kata baku sesuai slang mapping
    filtered = [w for w in normalized if w not in stop_words_custom] # menghpus kata yang termasuk stopword
    return " ".join(filtered)

df['cleaned_text'] = df['content'].apply(clean_text)

# menghapus data kosong
df = df[df['cleaned_text'].str.strip() != ''].copy()

# menghapus teks yang sama tapi punya label berbeda (teks kontradiktif)
jumlah_label = df.groupby('cleaned_text')['label'].nunique()
teks_konflik = jumlah_label[jumlah_label > 1].index
df = df[~df['cleaned_text'].isin(teks_konflik)]
df = df.drop_duplicates(subset=['cleaned_text']).copy()

print(f"Total data setelah dibersihkan: {len(df)}")
display(df[['content', 'cleaned_text', 'label']].head())

Total data setelah dibersihkan: 27801


,content,cleaned_text,label
0,aplikasi GO block driver terdekat juga banyak ...,aplikasi go block driver terdekat ngapain ngas...,Negatif
1,pengiriman murah cuma ujung²nya ga dianter sur...,pengiriman murah ujung nya tidak dianter suruh...,Negatif
2,apaan mau up&rade pake foto ktp dan muka,up rade pakai foto ktp muka,Negatif
3,"aplikasi aneh, ada gratis ongkir tapi suruh ba...",aplikasi aneh gratis ongkos kirim tapi suruh b...,Negatif
4,gojek sekarang kenapa ya untuk mencari cepat t...,gojek ya mencari cepat tapi lokasi si driver s...,Negatif


### 4. Noise Filtering Menggunakan Confidence Score
Melakukan seleksi per kelas dengan *Stratified K-Fold* dan *Logistic Regression* untuk menyaring label bertingkat noise tinggi (*label noise filtering*).

In [64]:
# Melakukan filtering label noise menggunakan skor probabilitas (confidence score) dari Logistic Regression
KEEP_RATIO = 0.6

tfidf_filter = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
x_filter = tfidf_filter.fit_transform(df['cleaned_text'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(LogisticRegression(C=3, max_iter=1000),
                          x_filter, df['label'], cv=skf, method='predict_proba')

kelas = sorted(df['label'].unique())   # ['Negatif', 'Netral', 'Positif']
idx_label = df['label'].map({k: i for i, k in enumerate(kelas)}).values
df['confidence'] = proba[np.arange(len(df)), idx_label]

bagian = []
for label, grup in df.groupby('label'):
    bagian.append(grup.nlargest(int(len(grup) * KEEP_RATIO), 'confidence'))
df_bersih = pd.concat(bagian).reset_index(drop=True)

print(f"Data sebelum filter: {len(df)} | sesudah filter: {len(df_bersih)}")
print(df_bersih['label'].value_counts())

Data sebelum filter: 27801 | sesudah filter: 16679
label
Negatif    8005
Positif    4863
Netral     3811
Name: count, dtype: int64


### 5. Pembagian Data, Tokenisasi, dan Penanganan Class Weight
Melakukan pembagian dataset (*train/test split*) 80/20, tokenisasi teks, serta perhitungan *class weight* untuk menyeimbangkan perhatian model secara matematis tanpa duplikasi data.

In [65]:
# mengecek panjang teks untuk menentukan MAX_LEN
panjang = df_bersih['cleaned_text'].apply(lambda x: len(x.split()))
print(np.percentile(panjang, [50, 90, 95, 99]))

# melakukan split data 80/20 (stratified)
label_encoder = LabelEncoder()
df_bersih['label_int'] = label_encoder.fit_transform(df_bersih['label'])

x_train_text, x_test_text, y_train_int, y_test_int = train_test_split(
    df_bersih['cleaned_text'].values, df_bersih['label_int'].values,
    test_size=0.2, random_state=42, stratify=df_bersih['label_int'].values
)
y_train_raw = to_categorical(y_train_int, num_classes=3)
y_test = to_categorical(y_test_int, num_classes=3)

# melakukan tokenisasi (tokenizer hanya fit di data training)
VOCAB_SIZE = 10000
MAX_LEN = 50

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer.fit_on_texts(x_train_text)
x_train_padded = pad_sequences(tokenizer.texts_to_sequences(x_train_text), maxlen=MAX_LEN, padding='post', truncating='post')
x_test_padded = pad_sequences(tokenizer.texts_to_sequences(x_test_text), maxlen=MAX_LEN, padding='post', truncating='post')

# fitur TF-IDF untuk skema 1
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
x_train_tfidf = tfidf.fit_transform(x_train_text).toarray().astype('float32')
x_test_tfidf = tfidf.transform(x_test_text).toarray().astype('float32')

print(f"Data Training: {len(x_train_text)} | Data Testing: {len(x_test_text)}")
print("Kelas:", label_encoder.classes_)

[ 9. 28. 36. 51.]
Data Training: 13343 | Data Testing: 3336
Kelas: ['Negatif' 'Netral' 'Positif']


In [66]:
# class weights supaya kelas Netral mendapat perhatian ekstra
def hitung_class_weight(y_int):
    bobot = compute_class_weight(class_weight='balanced', classes=np.unique(y_int), y=y_int)
    return dict(enumerate(bobot))

def buat_callbacks():
    return [EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=0.0001)]

class_weight_dict = hitung_class_weight(y_train_int)
print(f"Kamus Bobot Kelas (Class Weights): \n{class_weight_dict}")

Kamus Bobot Kelas (Class Weights): 
{0: np.float64(0.6945138455132209), 1: np.float64(1.458729638132721), 2: np.float64(1.1433590402742073)}


### 6. Skema Percobaan 1: Dense MLP + TF-IDF (Split 80/20)
Pelatihan model skema 1 menggunakan arsitektur *Multi-Layer Perceptron (Dense)* dengan input ekstraksi fitur TF-IDF.

In [67]:
# pembuatan model1 menggunakan TF-IDF dan dense layer dengan split 80/20
model1 = Sequential([
    Input(shape=(x_train_tfidf.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])
model1.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

print("===== TRAINING SCHEMA 1 =====")
history1 = model1.fit(
    x_train_tfidf, y_train_raw,
    epochs=15, batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=buat_callbacks()
)

train_loss1, train_acc1 = model1.evaluate(x_train_tfidf, y_train_raw, verbose=0)
test_loss1, test_acc1 = model1.evaluate(x_test_tfidf, y_test, verbose=0)
print(f"\nAkurasi Training Skema 1 adalah {train_acc1 * 100:.2f}%")
print(f"Akurasi Testing Skema 1 adalah {test_acc1 * 100:.2f}%")

===== TRAINING SCHEMA 1 =====
Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 21ms/step - accuracy: 0.6582 - loss: 0.8946 - val_accuracy: 0.7903 - val_loss: 0.5387 - learning_rate: 5.0000e-04
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8317 - loss: 0.4348 - val_accuracy: 0.8614 - val_loss: 0.3451 - learning_rate: 5.0000e-04
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.8918 - loss: 0.2858 - val_accuracy: 0.8704 - val_loss: 0.3046 - learning_rate: 5.0000e-04
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9305 - loss: 0.2059 - val_accuracy: 0.8644 - val_loss: 0.3058 - learning_rate: 5.0000e-04
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 3s 13ms/step - accuracy: 0.9455 - loss: 0.1586 - val_accuracy: 0.8652 - val_loss: 0.3091 - learning_rate: 5.0000e-04
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 20ms/step - accuracy: 0.9629 - loss: 0.1209 - val_accuracy: 0.8712 - val_loss: 0.3098 - learning_rate: 1.0000e-04
Epoch 7/15
188/188 ━━━━━━━━━

### 7. Skema Percobaan 2: Bidirectional LSTM + Embedding (Split 80/20)
Pelatihan model utama skema 2 menggunakan arsitektur *Bidirectional LSTM (BiLSTM)* dengan layer *Embedding*.

In [68]:
# pembuatan model2 menggunakan Embedding dan Bidirectional LSTM dengan split 80/20
model2 = Sequential([
    Embedding(VOCAB_SIZE, 64, mask_zero=True),
    SpatialDropout1D(0.2),
    Bidirectional(LSTM(64, dropout=0.2)),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
model2.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

print("===== TRAINING SCHEMA 2 =====")
history2 = model2.fit(
    x_train_padded, y_train_raw,
    epochs=12, batch_size=32,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=buat_callbacks()
)

train_loss2, train_acc2 = model2.evaluate(x_train_padded, y_train_raw, verbose=0)
test_loss2, test_acc2 = model2.evaluate(x_test_padded, y_test, verbose=0)
print(f"\nAkurasi Training Skema 2 adalah {train_acc2 * 100:.2f}%")
print(f"Akurasi Testing Skema 2 adalah {test_acc2 * 100:.2f}%")

===== TRAINING SCHEMA 2 =====
Epoch 1/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 38s 83ms/step - accuracy: 0.6730 - loss: 0.7149 - val_accuracy: 0.8045 - val_loss: 0.4556 - learning_rate: 5.0000e-04
Epoch 2/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 40s 81ms/step - accuracy: 0.8424 - loss: 0.3883 - val_accuracy: 0.8120 - val_loss: 0.4228 - learning_rate: 5.0000e-04
Epoch 3/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 31s 82ms/step - accuracy: 0.8992 - loss: 0.2669 - val_accuracy: 0.8524 - val_loss: 0.3770 - learning_rate: 5.0000e-04
Epoch 4/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 32s 85ms/step - accuracy: 0.9258 - loss: 0.2017 - val_accuracy: 0.8487 - val_loss: 0.3951 - learning_rate: 5.0000e-04
Epoch 5/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 42s 87ms/step - accuracy: 0.9400 - loss: 0.1676 - val_accuracy: 0.8382 - val_loss: 0.4219 - learning_rate: 5.0000e-04
Epoch 6/12
376/376 ━━━━━━━━━━━━━━━━━━━━ 41s 85ms/step - accuracy: 0.9597 - loss: 0.1189 - val_accuracy: 0.8479 - val_loss: 0.4486 - learning_rate: 1.0000e-04
Epoch 7/12
376/376 ━━━

### 8. Skema Percobaan 3: Conv1D + Embedding (Split 70/30)
Pelatihan model pembanding skema 3 menggunakan arsitektur *Convolutional Neural Network 1D (Conv1D)* dengan variasi pembagian data 70/30.

In [69]:
# pembuatan model3 menggunakan Embedding dan Conv1D dengan split 70/30

# melakukan split data 70/30
x_train_text3, x_test_text3, y_train_int3, y_test_int3 = train_test_split(
    df_bersih['cleaned_text'].values, df_bersih['label_int'].values,
    test_size=0.3, random_state=42, stratify=df_bersih['label_int'].values
)
y_train_raw3 = to_categorical(y_train_int3, num_classes=3)
y_test3 = to_categorical(y_test_int3, num_classes=3)

# melakukan tokenizer
tokenizer3 = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
tokenizer3.fit_on_texts(x_train_text3)
x_train_padded3 = pad_sequences(tokenizer3.texts_to_sequences(x_train_text3), maxlen=MAX_LEN, padding='post', truncating='post')
x_test_padded3 = pad_sequences(tokenizer3.texts_to_sequences(x_test_text3), maxlen=MAX_LEN, padding='post', truncating='post')
class_weight_dict3 = hitung_class_weight(y_train_int3)

# pembuatan model3 menggunakan Embedding dan Conv1D
model3 = Sequential([
    Embedding(VOCAB_SIZE, 64),
    SpatialDropout1D(0.2),
    Conv1D(128, 5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(3, activation='softmax')
])
model3.compile(optimizer=Adam(learning_rate=0.0005), loss='categorical_crossentropy', metrics=['accuracy'])

print("===== TRAINING SCHEMA 3 =====")
history3 = model3.fit(
    x_train_padded3, y_train_raw3,
    epochs=12, batch_size=64,
    validation_split=0.1,
    class_weight=class_weight_dict3,
    callbacks=buat_callbacks()
)

train_loss3, train_acc3 = model3.evaluate(x_train_padded3, y_train_raw3, verbose=0)
test_loss3, test_acc3 = model3.evaluate(x_test_padded3, y_test3, verbose=0)
print(f"\nAkurasi Training Skema 3 adalah {train_acc3 * 100:.2f}%")
print(f"Akurasi Testing Skema 3 adalah {test_acc3 * 100:.2f}%")

===== TRAINING SCHEMA 3 =====
Epoch 1/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.5736 - loss: 0.9052 - val_accuracy: 0.7029 - val_loss: 0.6924 - learning_rate: 5.0000e-04
Epoch 2/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 6s 34ms/step - accuracy: 0.7126 - loss: 0.6207 - val_accuracy: 0.7329 - val_loss: 0.5672 - learning_rate: 5.0000e-04
Epoch 3/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 32ms/step - accuracy: 0.8062 - loss: 0.4485 - val_accuracy: 0.7723 - val_loss: 0.4841 - learning_rate: 5.0000e-04
Epoch 4/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step - accuracy: 0.8667 - loss: 0.3338 - val_accuracy: 0.8074 - val_loss: 0.4336 - learning_rate: 5.0000e-04
Epoch 5/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 6s 37ms/step - accuracy: 0.9051 - loss: 0.2475 - val_accuracy: 0.8116 - val_loss: 0.4604 - learning_rate: 5.0000e-04
Epoch 6/12
165/165 ━━━━━━━━━━━━━━━━━━━━ 9s 30ms/step - accuracy: 0.9307 - loss: 0.1873 - val_accuracy: 0.8219 - val_loss: 0.4699 - learning_rate: 5.0000e-04
Epoch 7/12
165/165 ━━━━━━━━━

### 9. Rangkuman Perbandingan 3 Skema dan Evaluasi Classification Report
Evaluasi menyeluruh membandingkan performa ketiga skema serta menampilkan laporan presisi, recall, dan F1-Score per kelas sentimen pada model terbaik.

In [70]:
# manmpilkan rangkuman dan evaluasi perbandingan dari 3 skema
results_df = pd.DataFrame({
    'Skema Percobaan': ['Skema 1', 'Skema 2', 'Skema 3'],
    'Algoritma': ['Dense (MLP)', 'BiLSTM', 'Conv1D'],
    'Ekstraksi Fitur': ['TF-IDF', 'Embedding', 'Embedding'],
    'Pembagian Data': ['80/20', '80/20', '70/30'],
    'Akurasi Training': [train_acc1 * 100, train_acc2 * 100, train_acc3 * 100],
    'Akurasi Testing': [test_acc1 * 100, test_acc2 * 100, test_acc3 * 100]
})
print("===== HASIL PERBANDINGAN 3 SKEMA PERCOBAAN =====")
display(results_df)

# laporan per kelas (cek recall kelas Netral)
y_pred = np.argmax(model1.predict(x_test_tfidf, verbose=0), axis=1)
print(classification_report(y_test_int, y_pred, target_names=label_encoder.classes_))

===== HASIL PERBANDINGAN 3 SKEMA PERCOBAAN =====


,Skema Percobaan,Algoritma,Ekstraksi Fitur,Pembagian Data,Akurasi Training,Akurasi Testing
0,Skema 1,Dense (MLP),TF-IDF,80/20,94.146746,87.440050
1,Skema 2,BiLSTM,Embedding,80/20,93.697071,86.300957
2,Skema 3,Conv1D,Embedding,70/30,91.400427,81.294966


              precision    recall  f1-score   support

     Negatif       0.91      0.87      0.89      1601
      Netral       0.71      0.79      0.75       762
     Positif       0.95      0.95      0.95       973

    accuracy                           0.87      3336
   macro avg       0.86      0.87      0.86      3336
weighted avg       0.88      0.87      0.88      3336



### 10. Pengujian Inferensi & Penyimpanan Artefak Model
Menguji model dengan kalimat ulasan baru dan menyimpan seluruh artefak model (model file, tokenizer, dataset bersih) untuk keperluan pengumpulan submission.

In [71]:
# pengujian kalimat sentimen baru
import pickle

def predict_text(text, model, tokenizer, label_encoder):
    cleaned = clean_text(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')
    pred = model.predict(padded, verbose=0)
    return label_encoder.inverse_transform([np.argmax(pred)])[0], float(np.max(pred))

contoh_kalimat = [
    "Aplikasi ini bagus banget, fiturnya lengkap dan transaksinya cepat!",
    "Biasa aja sih aplikasinya, kadang lancar kadang agak lambat.",
    "Jelek banget, sering error pas mau bayar dan koin saya hilang!"
]

print("\n===== BUKTI PENGUJIAN MODEL =====")
for teks in contoh_kalimat:
    hasil, conf = predict_text(teks, model2, tokenizer, label_encoder)
    print(f"Teks: {teks}")
    print(f"Hasil: {hasil} (confidence {conf:.2f})\n")

# simpan artefak
model2.save('model_bilstm.keras')
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)
df_bersih.to_csv('dataset_gojek_clean.csv', index=False)


===== BUKTI PENGUJIAN MODEL =====
Teks: Aplikasi ini bagus banget, fiturnya lengkap dan transaksinya cepat!
Hasil: Positif (confidence 1.00)

Teks: Biasa aja sih aplikasinya, kadang lancar kadang agak lambat.
Hasil: Positif (confidence 0.73)

Teks: Jelek banget, sering error pas mau bayar dan koin saya hilang!
Hasil: Negatif (confidence 0.98)

